# 09. User-Defined Functions (UDFs) & Vectorized Pandas UDFs

While PySpark has hundreds of built-in functions, you will occasionally need custom business logic. That is where **UDFs** come in.

### Topics Covered:
1. **What is a UDF? When should (and shouldn't) you use one?**
2. **Standard Python UDFs (`udf()`, `@udf` decorator)**
3. **Registering UDFs for Spark SQL**
4. **The Performance Bottleneck of Standard Python UDFs (JVM $\leftrightarrow$ Python Serialization)**
5. **Pandas UDFs (Vectorized UDFs via Apache Arrow) - 10x-100x Faster!**
6. **Best Practice Hierarchy for Writing Logic in PySpark**
7. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [1]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, DoubleType

spark = SparkSession.builder \
    .appName("09_UDFs_and_Pandas_UDFs") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 18:56:41 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/11 18:56:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 18:56:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready!


---
## 2. When Should You Use a UDF?

### The Golden Rule of PySpark Functions:
1. **1st Choice: Built-in PySpark Functions (`pyspark.sql.functions`)**
   * Always check here first! They execute directly inside the JVM in optimized C++/Java bytecode.
2. **2nd Choice: Pandas UDFs (Vectorized UDFs with Apache Arrow)**
   * If custom logic is required, use Pandas UDFs because they process whole batches of rows at once.
3. **3rd Choice: Standard Python UDFs**
   * Use only when neither of the above works. Standard UDFs evaluate row-by-row and are significantly slower.


---
## 3. Creating a Standard Python UDF

Let's write a custom text masking function that masks credit card numbers, showing only the last 4 digits: `****-****-****-1234`.


In [2]:
# Step 1: Write standard Python function
def mask_card(card_number):
    if card_number is None:
        return None
    # Strip any dashes or spaces
    cleaned = str(card_number).replace("-", "").replace(" ", "")
    if len(cleaned) < 4:
        return "****"
    return f"****-****-****-{cleaned[-4:]}"

# Step 2: Convert to PySpark UDF by specifying return type
mask_card_udf = udf(mask_card, StringType())

# Sample customer data
cards_data = [
    (1, "Alice", "1234-5678-9012-3456"),
    (2, "Bob", "9876 5432 1098 7654"),
    (3, "Charlie", "4111222233334444"),
    (4, "Diana", None)
]

cards_df = spark.createDataFrame(cards_data, ["id", "customer", "card_number"])

# Step 3: Apply the UDF to a column
masked_df = cards_df.withColumn("masked_card", mask_card_udf(col("card_number")))
masked_df.show(truncate=False)


/mnt/d/Codes/AntiGravity/.venv/lib/python3.14/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


+---+--------+-------------------+-------------------+
|id |customer|card_number        |masked_card        |
+---+--------+-------------------+-------------------+
|1  |Alice   |1234-5678-9012-3456|****-****-****-3456|
|2  |Bob     |9876 5432 1098 7654|****-****-****-7654|
|3  |Charlie |4111222233334444   |****-****-****-4444|
|4  |Diana   |NULL               |NULL               |
+---+--------+-------------------+-------------------+



---
## 4. Using Python Decorator Syntax `@udf`

You can also use Python's `@udf` decorator directly on your function definition.


In [3]:
@udf(returnType=StringType())
def categorize_credit_score(score):
    if score is None:
        return "Unknown"
    elif score >= 750:
        return "Excellent"
    elif score >= 700:
        return "Good"
    elif score >= 600:
        return "Fair"
    else:
        return "Poor"

scores_df = spark.createDataFrame([
    (1, "Alice", 780),
    (2, "Bob", 640),
    (3, "Charlie", 710),
    (4, "Diana", 550)
], ["id", "name", "credit_score"])

scores_df.withColumn("rating", categorize_credit_score(col("credit_score"))).show()


+---+-------+------------+---------+
| id|   name|credit_score|   rating|
+---+-------+------------+---------+
|  1|  Alice|         780|Excellent|
|  2|    Bob|         640|     Fair|
|  3|Charlie|         710|     Good|
|  4|  Diana|         550|     Poor|
+---+-------+------------+---------+



---
## 5. Registering UDFs for Spark SQL

To use a UDF inside a `spark.sql("SELECT ...")` query, register it with `spark.udf.register()`.


In [4]:
# Register the function with Spark SQL catalog
spark.udf.register("sql_mask_card", mask_card, StringType())

cards_df.createOrReplaceTempView("cards_table")

spark.sql("""
    SELECT 
        customer,
        sql_mask_card(card_number) AS masked_card_sql
    FROM cards_table
""").show(truncate=False)


+--------+-------------------+
|customer|masked_card_sql    |
+--------+-------------------+
|Alice   |****-****-****-3456|
|Bob     |****-****-****-7654|
|Charlie |****-****-****-4444|
|Diana   |NULL               |
+--------+-------------------+



---
## 6. Why are Standard Python UDFs Slow?

```
┌────────────────────────────────┐
│   Spark JVM (Executor Memory)  │
└───────────────┬────────────────┘
                │ 1. Serialize row data (Pickle)
                ▼
┌────────────────────────────────┐
│   Python Worker Subprocess     │  <-- Slow row-by-row Python loop
└───────────────┬────────────────┘
                │ 2. Deserialize & send back
                ▼
┌────────────────────────────────┐
│   Spark JVM (Executor Memory)  │
└────────────────────────────────┘
```
For every single row:
1. Data must be copied out of the JVM and serialized into Python via Pickle.
2. The Python interpreter runs the function row-by-row.
3. The result is serialized and copied back into the JVM.
4. **Catalyst cannot optimize inside black-box Python code!**


---
## 7. Vectorized Pandas UDFs (Powered by Apache Arrow)

### How Pandas UDFs solve the problem:
* Instead of processing 1 row at a time, Apache Spark transfers an entire **batch of rows** (a Pandas Series) in-memory using **Apache Arrow** format (zero-copy memory transfer).
* The function receives a `pandas.Series` and returns a `pandas.Series`.
* Runs vectorized operations in C/NumPy speed: **Up to 100x faster!**


In [5]:
# Check if pandas and pyarrow are available
try:
    import pandas as pd
    from pyspark.sql.functions import pandas_udf

    # Series to Series Pandas UDF: Calculate compound interest
    @pandas_udf(DoubleType())
    def calculate_compound_interest(principal: pd.Series, rate: pd.Series, years: pd.Series) -> pd.Series:
        # Vectorized math directly on Pandas Series!
        return principal * ((1 + rate) ** years)

    investments = spark.createDataFrame([
        (1000.0, 0.05, 5),
        (5000.0, 0.07, 10),
        (2500.0, 0.04, 3)
    ], ["principal", "rate", "years"])

    result_df = investments.withColumn(
        "future_value", 
        calculate_compound_interest(col("principal"), col("rate"), col("years"))
    )
    print("Pandas UDF Result (Vectorized with Apache Arrow):")
    result_df.show()

except ImportError:
    print("Note: pandas/pyarrow not installed in this environment. Pandas UDF concept demonstrated above.")


Note: pandas/pyarrow not installed in this environment. Pandas UDF concept demonstrated above.


---
## 8. Summary of Notebook 09
* Always try to use **built-in PySpark functions** (`pyspark.sql.functions`) first.
* Standard Python UDFs (`@udf`) evaluate row-by-row and have high serialization overhead between JVM and Python.
* You can register UDFs with `spark.udf.register()` for Spark SQL.
* **Pandas UDFs** (`@pandas_udf`) use Apache Arrow for high-speed batch vectorized processing.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [6]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")


SparkSession stopped successfully! RAM released.
